# SentinelAI — Notebook 09: Coding Worker

**Purpose:** Explain how `workers/coding.py` works — the queue-based worker
that wraps the full three-stage pipeline into a production-ready process.

**Prerequisites:** Notebooks 05, 06, 07 (hybrid search, reranker, LLM coder)
must be understood first. This notebook explains only the *orchestration layer*
on top of those components.

---

## What Problem Does the Worker Solve?

Notebooks 05-07 showed how to code **one report** interactively.
In production we have thousands of MAUDE reports that need coding.

The coding worker solves three problems that don't exist in a notebook:

1. **Scale** — iterate over all uncoded reports automatically
2. **Durability** — if one report fails, the worker logs the error and continues (not crash)
3. **Resume-ability** — if the worker is stopped and restarted, it picks up where it left off
   (never re-codes a report that already has a result)

---

## Architecture Overview

```
raw.maude_reports
        |
        |  fetch_uncoded_reports()  -- SQL: NOT EXISTS coding_results
        v
  [ report batch ]
        |
        |  code_report()  -- three stages per report
        v
  Stage 1: HybridSearcher        -> Top-20 MedDRA candidates
  Stage 2: CrossEncoderReranker  -> Top-5 by relevance score
  Stage 3: LLMCoder (Ollama)     -> Final PT code + rationale
        |
        |  write_coding_result()
        v
processed.coding_results
```

The worker runs a **polling loop**: it keeps fetching and coding batches until
either:
- the queue is empty (`--once` mode: exit; continuous mode: sleep 60s and check again)
- the `--limit` cap is reached
- the user sends SIGINT (Ctrl+C)

---

## Part 1: How Reports Are Selected

### `fetch_uncoded_reports()` — the queue query

The worker doesn't use an explicit task queue (like Redis or RabbitMQ).
Instead, it queries the database directly:

```sql
SELECT r.mdr_report_key, r.mdr_text, r.product_code, r.date_received
FROM raw.maude_reports r
WHERE r.mdr_text IS NOT NULL
  AND r.mdr_text <> ''
  AND NOT EXISTS (
      SELECT 1
      FROM processed.coding_results c
      WHERE c.mdr_report_key = r.mdr_report_key
  )
ORDER BY r.date_received DESC NULLS LAST
LIMIT %(batch_size)s
```

**Key design decisions:**

- **`NOT EXISTS` instead of `NOT IN`** — `NOT IN` on a large subquery is slow and
  behaves unexpectedly with NULLs. `NOT EXISTS` is the correct pattern here.
- **`ORDER BY date_received DESC`** — newest reports are coded first. The most recent
  adverse events are usually most relevant for signal detection.
- **`mdr_text IS NOT NULL AND mdr_text <> ''`** — skip reports with no narrative;
  there's nothing for the pipeline to process.

### Resume-ability

Because we only fetch reports that have **no existing coding result**, the worker
is naturally resume-able:

- Stop the worker at any time (e.g. SSH session closes)
- Restart it later
- It picks up from the next uncoded report

No separate checkpoint file is needed.

---

## Part 2: Per-Report Coding (`code_report()`)

Each report goes through the same three stages as in Notebooks 05-07:

```python
# Stage 1
candidates = searcher.search(narrative, top_k=20)

# Stage 2
reranked = reranker.rerank(narrative, candidates, top_k=5)

# Stage 3
llm_result = coder.code(narrative, reranked)
```

### Confidence scoring

Three confidence signals are combined into `final_confidence`:

| Signal | Source | Range | Weight |
|--------|--------|-------|--------|
| CrossEncoder score | Stage 2 (sigmoid-normalized) | 0 to 1 | 30% |
| LLM confidence | Stage 3 (self-reported) | 0 to 1 | 70% |

**Why sigmoid on the CrossEncoder score?**

The CrossEncoder (`ms-marco-MiniLM-L-6-v2`) outputs a raw logit — an
unbounded real number (e.g. `-3.2` to `+4.8`). This is **not** a probability.
We apply sigmoid to map it to (0, 1):

```
sigmoid(x) = 1 / (1 + e^(-x))
```

Examples:
- logit = 0.0  -> sigmoid = 0.50 (uncertain)
- logit = 2.0  -> sigmoid = 0.88 (fairly confident)
- logit = -2.0 -> sigmoid = 0.12 (likely wrong)

**Why 70% weight on LLM confidence?**

The LLM sees the full narrative AND all five candidates before choosing.
Its confidence signal integrates more information than the CrossEncoder's
pairwise score. Empirically, LLM confidence correlates better with actual
correctness — but it can hallucinate, so we don't weight it 100%.

**When LLM is unavailable:**

If `--skip-llm` is set, or if the Ollama call fails at runtime:
- `llm_confidence` is stored as NULL
- `final_confidence = sigmoid(crossencoder_score)` (CE score alone)
- The CrossEncoder's top result is used as the final PT

### Reports flagged for human review

Any result where `final_confidence < 0.5` is written with that low confidence
score. The signal detection module (Module 3) queries these with:

```sql
SELECT * FROM processed.coding_results WHERE final_confidence < 0.5
```

This queue is what a human reviewer would work through to improve coding quality.

---

## Part 3: Error Handling

### Per-report error isolation

Each report is processed in its own `try/except` block:

```python
for report in reports:
    try:
        result = code_report(report, searcher, reranker, coder)
        write_coding_result(conn, result)
        conn.commit()
    except Exception as exc:
        conn.rollback()
        logger.error("[ERR] %s: %s", rk, exc)
```

If one report causes an exception (e.g. malformed narrative, DB timeout),
the error is logged and the worker moves to the next report. The batch
is not aborted.

### Transaction management

Each report is committed individually — not as a batch commit. This means:

- **If the worker crashes mid-batch**, only the reports already committed are
  in `coding_results`. The others will be picked up on the next run.
- **No partial writes** — either the full result for a report is in the DB, or none of it.

### LLM fallback

The LLM stage can fail independently without failing the whole report:

```python
try:
    llm_result = coder.code(narrative, reranked)
    llm_confidence = llm_result.confidence
except Exception as exc:
    logger.warning("LLM call failed for %s (%s), falling back to CE", rk, exc)
    # llm_confidence stays None -> sigmoid(ce_score) is used as final_confidence
```

### DB connection management

A **new DB connection is opened per batch** (not per report, not reused forever):

```python
while True:                     # polling loop
    conn = get_connection()     # fresh connection for this batch
    try:
        ...                     # process reports
    finally:
        conn.close()            # always close, even on error
```

Why per-batch and not one connection for the whole run?

In a long-running worker (hours/days), DB connections can become stale
if the server restarts or the SSH tunnel closes. Opening a fresh connection
per batch means the worker self-heals on the next batch after a connection issue.

---

## Part 4: Model Loading

Models are loaded **once at startup** and reused across all batches:

```python
embedding_model = EmbeddingModel()       # PubMedBERT -- ~440MB, ~10s
searcher = HybridSearcher(conn, ...)     # wraps EmbeddingModel + DB connection
reranker = CrossEncoderReranker()        # MiniLM-L-6 -- ~22MB, ~2s
coder    = LLMCoder(...)                 # Ollama REST client -- no model file locally
```

**Why not reload per batch?**

PubMedBERT (440MB) takes ~10 seconds to load on CPU and another ~5 seconds
to warm up. With batches of 25 reports at ~5 seconds each, reloading per
batch would add 15 seconds of overhead (37% of batch time) for zero benefit.

**DB connection injection for HybridSearcher**

`HybridSearcher` holds a reference to a psycopg2 connection. Since we open
a fresh connection per batch, we update the searcher's connection reference
before each batch:

```python
conn = get_connection()
searcher.conn = conn     # inject new connection
```

The model stays the same; only the DB connection is replaced.

---

## Part 5: Running the Worker

### Prerequisites

SSH tunnels must be open:

```bash
# Terminal 1 -- PostgreSQL
ssh -L 5432:localhost:5432 cap@46.225.109.99

# Terminal 2 -- Ollama (skip if using --skip-llm)
ssh -L 11434:localhost:11434 cap@46.225.109.99
```

### Common run patterns

```bash
# Continuous mode (production): code everything, keep polling
python -m vigilex.workers.coding

# Test run: code 50 insulin pump reports, then exit
python -m vigilex.workers.coding --once --product-code LZG --limit 50

# No Ollama (stages 1+2 only)
python -m vigilex.workers.coding --skip-llm --once

# Debug: tiny batches + verbose logging
python -m vigilex.workers.coding --batch-size 5 --limit 10 --once --verbose
```

### Expected output

```
2026-04-23 10:31:42 | INFO | vigilex.workers.coding | === SentinelAI Coding Worker starting ===
2026-04-23 10:31:42 | INFO | vigilex.workers.coding | Loading PubMedBERT embedding model...
2026-04-23 10:31:53 | INFO | vigilex.workers.coding | PubMedBERT loaded on cpu in 10.7s
2026-04-23 10:31:53 | INFO | vigilex.workers.coding | Loading CrossEncoder reranker (MiniLM-L-6)...
2026-04-23 10:31:55 | INFO | vigilex.workers.coding | CrossEncoder loaded in 1.8s
2026-04-23 10:31:55 | INFO | vigilex.workers.coding | Initialising LLM coder (Ollama at http://localhost:11434)...
2026-04-23 10:31:55 | INFO | vigilex.workers.coding | Ollama reachable -- models: ['llama3.2:3b']
2026-04-23 10:31:55 | INFO | vigilex.workers.coding | Fetched 25 reports for coding (1847 still pending in DB).
2026-04-23 10:31:59 | INFO | vigilex.workers.coding | [OK] 8560781-2024-00123  PT=Hypoglycaemia              conf=0.847  4231ms
2026-04-23 10:32:04 | INFO | vigilex.workers.coding | [OK] 8560781-2024-00124  PT=Pump malfunction           conf=0.712  4876ms
...
2026-04-23 10:34:22 | INFO | vigilex.workers.coding | ==> Batch done: 25 OK / 0 errors | Running total: 25
```

### Throughput estimate

On a CPU-only machine with Ollama running locally:

| Stage | Time per report |
|-------|----------------|
| Hybrid search | ~200ms |
| CrossEncoder rerank | ~150ms |
| LLM (llama3.2:3b) | ~3-8s |
| DB write + commit | ~20ms |
| **Total** | **~4-9s** |

For 1,000 reports: roughly **1.5 to 2.5 hours** on CPU.
With GPU or a larger Ollama model this would be faster.

---

## Part 6: Verifying Results in the DB

After running the worker, check results directly in PostgreSQL:

```sql
-- How many reports have been coded?
SELECT COUNT(*) FROM processed.coding_results;

-- Confidence distribution
SELECT
    ROUND(final_confidence::numeric, 1) AS conf_bucket,
    COUNT(*) AS n_reports
FROM processed.coding_results
GROUP BY 1
ORDER BY 1;

-- Reports flagged for human review (low confidence)
SELECT mdr_report_key, pt_name, final_confidence
FROM processed.coding_results
WHERE final_confidence < 0.5
ORDER BY final_confidence;

-- Most common PT codes assigned
SELECT pt_name, COUNT(*) AS n
FROM processed.coding_results
GROUP BY pt_name
ORDER BY n DESC
LIMIT 20;

-- Full view via the helper view (joins MAUDE report + coding result)
SELECT
    s.mdr_report_key,
    s.product_code,
    s.mdr_text[:100] AS narrative_excerpt,
    s.pt_name,
    s.soc_name,
    s.final_confidence
FROM processed.search_base s
WHERE s.pt_name IS NOT NULL
ORDER BY s.date_received DESC
LIMIT 20;
```

---

## Part 7: Docker Deployment

The coding worker is containerized via `docker/Dockerfile.worker-coding`.
The image includes torch + transformers for PubMedBERT (build time ~6-10min,
then cached).

Entry point: `python -m vigilex.workers.coding`

On the Hetzner server, the worker container can be started via Portainer
(`http://46.225.109.99:9000`) or `docker compose up worker-coding`.

In continuous mode it will run indefinitely, processing new reports as they
arrive from the ingestion worker.

### Container pipeline flow

```
worker-ingest         worker-coding           Ollama
     |                     |                    |
     | raw.maude_reports -> |                    |
     |                     | -- Stage 1+2 ----> |  (local inference)
     |                     | <-- Stage 3 ------ |
     |                     |
     |            processed.coding_results
```

All inference stays on the Hetzner server — no PHI leaves the infrastructure.

---

## Summary: Module 2 Complete

The MedDRA Coding Engine (Module 2) is now fully built:

| Component | File | Status |
|-----------|------|--------|
| Stage 1: Hybrid Search | `src/vigilex/coding/hybrid_search.py` | Done |
| Stage 2: CrossEncoder Reranker | `src/vigilex/coding/reranker.py` | Done |
| Stage 3: LLM Coder | `src/vigilex/coding/llm_coder.py` | Done |
| Coding Worker | `src/vigilex/workers/coding.py` | Done |
| Smoke test | `scripts/smoke_test_pipeline.py` | Done |
| Debugging diary | `notebooks/08_pipeline_debugging.ipynb` | Done |

**Next: Module 3** — PRR/ROR signal detection over the coded reports,
Grafana dashboard, and Streamlit frontend for interactive review.